03/14/2026 | 
Author: Cristian Ortega Singer | 
Mail: cris.ortega@fau.de


Load dependencies

In [1]:
from pathlib import Path
import subprocess
import pandas as pd
import json
import shutil
from datetime import datetime

Setup variables and presets

In [ ]:
SOURCE_ROOT = Path("downloads")
OUTPUT_ROOT = Path("normalized_mp4")

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True, parents=True)

# Allowed input formats
VIDEO_EXTENSIONS = {".mkv", ".webm", ".mp4"}

# Dry run = print commands only, do not process files
DRY_RUN = True
# Limit processing for testing; set to None for full corpus
LIMIT = None #5 None
# Overwrite existing outputs?
OVERWRITE = False

# ffmpeg settings
VIDEO_CODEC = "libx264"
AUDIO_CODEC = "aac"
AUDIO_BITRATE = "192k"
CRF = 18
PRESET = "medium"

# Loudness normalization target
# EBU R128 style targets
LOUDNORM_I = -16
LOUDNORM_LRA = 11
LOUDNORM_TP = -1.5

# Resolution settings
MAX_LONG_EDGE = 854
DOWNSCALE_ONLY = True
#Frame rate normalization
TARGET_FPS = 25

# Logging
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
PROCESS_LOG = LOG_DIR / f"conversion_log_{TIMESTAMP}.txt"
INVENTORY_CSV = LOG_DIR / f"corpus_inventory_{TIMESTAMP}.csv"
RESULTS_CSV = LOG_DIR / f"conversion_results_{TIMESTAMP}.csv"

Check for ffmpeg on path

In [3]:
def check_ffmpeg():
    ffmpeg_path = shutil.which("ffmpeg")
    ffprobe_path = shutil.which("ffprobe")
    
    print("ffmpeg:", ffmpeg_path)
    print("ffprobe:", ffprobe_path)
    
    if ffmpeg_path is None:
        raise RuntimeError("ffmpeg not found in PATH.")
    if ffprobe_path is None:
        raise RuntimeError("ffprobe not found in PATH.")
        
check_ffmpeg()

ffmpeg: C:\Users\CrisO\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin\ffmpeg.EXE
ffprobe: C:\Users\CrisO\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin\ffprobe.EXE


Establish helper functions

In [ ]:
def find_video_files(source_root: Path, allowed_exts=None):

    if allowed_exts is None:
        allowed_exts = VIDEO_EXTENSIONS
    
    files = []
    for path in source_root.rglob("*"):
        if path.is_file() and path.suffix.lower() in allowed_exts:
            files.append(path)
    return sorted(files)
 
def make_output_path(input_file: Path, source_root: Path, output_root: Path) -> Path:

    stem = input_file.stem
    output_file = output_root / f"{stem}.mp4"
    
    output_root.mkdir(parents=True, exist_ok=True)
    
    return output_file

def probe_streams(input_file: Path):

    # Use ffprobe to get stream information
    cmd = [
        "ffprobe",
        "-v", "error",
        "-print_format", "json",
        "-show_streams",
        "-show_format",
        str(input_file)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        return None
    
    try:
        return json.loads(result.stdout)
    except json.JSONDecodeError:
        return None


def summarize_probe(probe_data):

    # summary from ffprobe output
    if probe_data is None:
        return {
            "duration": None,
            "size": None,
            "bit_rate": None,
            "video_codec": None,
            "audio_codec": None,
            "width": None,
            "height": None,
            "audio_channels": None,
            "sample_rate": None,
        }
    
    format_info = probe_data.get("format", {})
    streams = probe_data.get("streams", [])
    
    video_stream = next((s for s in streams if s.get("codec_type") == "video"), {})
    audio_stream = next((s for s in streams if s.get("codec_type") == "audio"), {})
    
    return {
        "duration": format_info.get("duration"),
        "size": format_info.get("size"),
        "bit_rate": format_info.get("bit_rate"),
        "video_codec": video_stream.get("codec_name"),
        "audio_codec": audio_stream.get("codec_name"),
        "width": video_stream.get("width"),
        "height": video_stream.get("height"),
        "audio_channels": audio_stream.get("channels"),
        "sample_rate": audio_stream.get("sample_rate"),
    }


def build_scale_filter(max_long_edge=1280, downscale_only=True):
    
    # preserves aspect ratio of videos    
    if downscale_only:
        return (
            f"scale="
            f"'if(gte(iw,ih),min({max_long_edge},iw),-2)':"
            f"'if(gte(iw,ih),-2,min({max_long_edge},ih))':"
            f"flags=lanczos"
        )
    else:
        return (
            f"scale="
            f"'if(gte(iw,ih),{max_long_edge},-2)':"
            f"'if(gte(iw,ih),-2,{max_long_edge})':"
            f"flags=lanczos"
        )

Create csv with technical infos from ffprobe for each video

In [5]:
video_files = find_video_files(SOURCE_ROOT)
print(f"Found {len(video_files)} video files.")

inventory_rows = []

for vf in video_files:
    probe = probe_streams(vf)
    summary = summarize_probe(probe)
    
    row = {
        "input_path": str(vf),
        "parent_folder": str(vf.parent),
        "stem": vf.stem,
        "suffix": vf.suffix.lower(),
        **summary
    }
    inventory_rows.append(row)

inventory_df = pd.DataFrame(inventory_rows)
inventory_df.to_csv(INVENTORY_CSV, index=False)

print(f"Saved inventory to: {INVENTORY_CSV}")
inventory_df.head()

Found 2352 video files.
Saved inventory to: logs\corpus_inventory_20260313_234545.csv


,input_path,parent_folder,stem,suffix,duration,size,bit_rate,video_codec,audio_codec,width,height,audio_channels,sample_rate
0,downloads\2012-11-15_wtsSeRrOBZQ\2012-11-15_wt...,downloads\2012-11-15_wtsSeRrOBZQ,2012-11-15_wtsSeRrOBZQ,.mp4,104.582676,22780922,1742615,h264,aac,1280.0,720.0,2.0,44100
1,downloads\2013-04-26_8HHUta7n0y4\2013-04-26_8H...,downloads\2013-04-26_8HHUta7n0y4,2013-04-26_8HHUta7n0y4,.mkv,334.601000,61740238,1476151,h264,opus,1920.0,1080.0,2.0,48000
2,downloads\2013-04-26_bzkLtFLpWEg\2013-04-26_bz...,downloads\2013-04-26_bzkLtFLpWEg,2013-04-26_bzkLtFLpWEg,.mp4,242.718186,9225356,304068,h264,aac,854.0,480.0,2.0,44100
3,downloads\2013-04-26_eDrIDaJRwJ0\2013-04-26_eD...,downloads\2013-04-26_eDrIDaJRwJ0,2013-04-26_eDrIDaJRwJ0,.mp4,338.895238,31983030,754995,h264,aac,854.0,480.0,2.0,44100
4,downloads\2013-04-26_OKz_pgqQycc\2013-04-26_OK...,downloads\2013-04-26_OKz_pgqQycc,2013-04-26_OKz_pgqQycc,.mp4,438.299864,32051313,585011,h264,aac,854.0,480.0,2.0,44100


Inspect results

In [6]:
print("File count by extension:")
display(inventory_df["suffix"].value_counts())

print("\nVideo codec distribution:")
display(inventory_df["video_codec"].value_counts(dropna=False).head(20))

print("\nAudio codec distribution:")
display(inventory_df["audio_codec"].value_counts(dropna=False).head(20))

print("\nSample rate distribution:")
display(inventory_df["sample_rate"].value_counts(dropna=False).head(20))

File count by extension:


suffix
.mkv     1643
.webm     613
.mp4       96
Name: count, dtype: int64


Video codec distribution:


video_codec
h264    1739
vp9      484
av1      126
NaN        3
Name: count, dtype: int64


Audio codec distribution:


audio_codec
opus    2253
aac       96
NaN        3
Name: count, dtype: int64


Sample rate distribution:


sample_rate
48000    2255
44100      94
NaN         3
Name: count, dtype: int64

In [ ]:
inventory_df["width"] = pd.to_numeric(inventory_df["width"], errors="coerce")
inventory_df["height"] = pd.to_numeric(inventory_df["height"], errors="coerce")
inventory_df["long_edge"] = inventory_df[["width", "height"]].max(axis=1)

print("Resolution summary:")
display(inventory_df["long_edge"].describe())

print("\nVideos above x long edge:")
display((inventory_df["long_edge"] > 1280).value_counts())

print("\nMost common resolutions:")
display(
    inventory_df.groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

Resolution summary:


count    2349.000000
mean     1931.633887
std       365.698243
min       480.000000
25%      1920.000000
50%      1920.000000
75%      1920.000000
max      3840.000000
Name: long_edge, dtype: float64


Videos above 1280 long edge:


long_edge
True     2243
False     109
Name: count, dtype: int64


Most common resolutions:


width   height
1920.0  1080.0    1871
1080.0  1920.0     307
3840.0  2160.0      54
1280.0  720.0       41
854.0   480.0       32
720.0   1280.0      18
640.0   480.0        8
3840.0  1634.0       6
640.0   360.0        4
360.0   640.0        2
270.0   480.0        1
1440.0  1080.0       1
1920.0  1020.0       1
        818.0        1
3840.0  1540.0       1
        1920.0       1
dtype: int64

ffmpeg builder

In [ ]:
def build_ffmpeg_command(input_file: Path, output_file: Path, overwrite=False):
 
    # conversion to mp4
    # resolution normalization
    # frame rate normalization
    # audio loudness normalization

    output_file.parent.mkdir(parents=True, exist_ok=True)
    overwrite_flag = "-y" if overwrite else "-n"

    loudnorm_filter = (
        f"loudnorm=I={LOUDNORM_I}:LRA={LOUDNORM_LRA}:TP={LOUDNORM_TP}"
    )

    scale_filter = build_scale_filter(
        max_long_edge=MAX_LONG_EDGE,
        downscale_only=DOWNSCALE_ONLY
    )

    video_filter = f"{scale_filter},fps={TARGET_FPS}"

    cmd = [
        "ffmpeg",
        overwrite_flag,
        "-i", str(input_file),

        "-map", "0:v:0?",
        "-map", "0:a:0?",

        "-vf", video_filter,

        "-c:v", VIDEO_CODEC,
        "-preset", PRESET,
        "-crf", str(CRF),

        "-c:a", AUDIO_CODEC,
        "-b:a", AUDIO_BITRATE,
        "-af", loudnorm_filter,

        "-movflags", "+faststart",

        str(output_file)
    ]

    return cmd

Preview of ffmpeg command call

In [19]:
sample_files = video_files[:LIMIT] if LIMIT is not None else video_files[:5]

for vf in sample_files:
    out = make_output_path(vf, SOURCE_ROOT, OUTPUT_ROOT)
    cmd = build_ffmpeg_command(vf, out, overwrite=OVERWRITE)
    print("INPUT :", vf)
    print("OUTPUT:", out)
    print("CMD   :", " ".join(cmd))
    print("-" * 80)

INPUT : downloads\2012-11-15_wtsSeRrOBZQ\2012-11-15_wtsSeRrOBZQ.mp4
OUTPUT: normalized_mp4\2012-11-15_wtsSeRrOBZQ.mp4
CMD   : ffmpeg -n -i downloads\2012-11-15_wtsSeRrOBZQ\2012-11-15_wtsSeRrOBZQ.mp4 -map 0:v:0? -map 0:a:0? -vf scale='if(gte(iw,ih),min(854,iw),-2)':'if(gte(iw,ih),-2,min(854,ih))':flags=lanczos,fps=25 -c:v libx264 -preset medium -crf 18 -c:a aac -b:a 192k -af loudnorm=I=-16:LRA=11:TP=-1.5 -movflags +faststart normalized_mp4\2012-11-15_wtsSeRrOBZQ.mp4
--------------------------------------------------------------------------------
INPUT : downloads\2013-04-26_8HHUta7n0y4\2013-04-26_8HHUta7n0y4.mkv
OUTPUT: normalized_mp4\2013-04-26_8HHUta7n0y4.mp4
CMD   : ffmpeg -n -i downloads\2013-04-26_8HHUta7n0y4\2013-04-26_8HHUta7n0y4.mkv -map 0:v:0? -map 0:a:0? -vf scale='if(gte(iw,ih),min(854,iw),-2)':'if(gte(iw,ih),-2,min(854,ih))':flags=lanczos,fps=25 -c:v libx264 -preset medium -crf 18 -c:a aac -b:a 192k -af loudnorm=I=-16:LRA=11:TP=-1.5 -movflags +faststart normalized_mp4\2013-0

Set up Process function

In [8]:
def run_command(cmd):
    return subprocess.run(cmd, capture_output=True, text=True)


def process_videos(
    files,
    source_root: Path,
    output_root: Path,
    dry_run=True,
    overwrite=False,
    log_file: Path | None = None
):
    results = []
    
    if log_file is not None:
        with open(log_file, "w", encoding="utf-8") as log:
            log.write(f"Processing started: {datetime.now().isoformat()}\n")
            log.write(f"Source root: {source_root}\n")
            log.write(f"Output root: {output_root}\n")
            log.write(f"Dry run: {dry_run}\n")
            log.write(f"Overwrite: {overwrite}\n\n")
    
    for i, input_file in enumerate(files, start=1):
        output_file = make_output_path(input_file, source_root, output_root)
        
        exists_already = output_file.exists()
        status = None
        returncode = None
        stderr_tail = None
        
        if exists_already and not overwrite:
            status = "skipped_exists"
        else:
            cmd = build_ffmpeg_command(input_file, output_file, overwrite=overwrite)
            
            if dry_run:
                status = "dry_run"
                returncode = 0
            else:
                result = run_command(cmd)
                returncode = result.returncode
                stderr_tail = result.stderr[-2000:] if result.stderr else None
                status = "ok" if result.returncode == 0 else "error"
        
        row = {
            "index": i,
            "input_file": str(input_file),
            "output_file": str(output_file),
            "status": status,
            "returncode": returncode,
            "output_exists": output_file.exists(),
            "stderr_tail": stderr_tail
        }
        results.append(row)
        
        print(f"[{i}/{len(files)}] {status}: {input_file.name}")
        
        if log_file is not None:
            with open(log_file, "a", encoding="utf-8") as log:
                log.write(json.dumps(row, ensure_ascii=False) + "\n")
    
    return pd.DataFrame(results)

Run pipeline

In [ ]:
input_files = video_files[:LIMIT] if LIMIT is not None else video_files
results_df = process_videos(
    files=input_files,
    source_root=SOURCE_ROOT,
    output_root=OUTPUT_ROOT,
    dry_run=False,
    overwrite=OVERWRITE,
    log_file=PROCESS_LOG
)

results_df.to_csv(RESULTS_CSV, index=False)

print(f"Saved processing results to: {RESULTS_CSV}")
print(f"Saved log to: {PROCESS_LOG}")

results_df.head()

[1/2352] skipped_exists: 2012-11-15_wtsSeRrOBZQ.mp4
[2/2352] skipped_exists: 2013-04-26_8HHUta7n0y4.mkv
[3/2352] skipped_exists: 2013-04-26_bzkLtFLpWEg.mp4
[4/2352] skipped_exists: 2013-04-26_eDrIDaJRwJ0.mp4
[5/2352] skipped_exists: 2013-04-26_OKz_pgqQycc.mp4
[6/2352] skipped_exists: 2013-04-26_r7exgls85kg.mkv
[7/2352] skipped_exists: 2013-07-03_-Bf375yCUP8.mp4
[8/2352] skipped_exists: 2013-07-03_-P2nVYjqaNc.mkv
[9/2352] skipped_exists: 2013-07-03_1K2NYb9Qg1o.mkv
[10/2352] skipped_exists: 2013-07-03_MUbvifkMXK8.mkv
[11/2352] skipped_exists: 2013-07-03_Nt7DeLYSzkk.mkv
[12/2352] skipped_exists: 2013-07-03_Ub-vVIXU4kE.mkv
[13/2352] skipped_exists: 2013-07-22_3249HNziLYQ.mkv
[14/2352] skipped_exists: 2013-07-22_fKMYi5MKaHw.mkv
[15/2352] skipped_exists: 2013-07-22_i8aoKtzoeYk.mkv
[16/2352] skipped_exists: 2013-07-22_IqBcwdvrpkY.mkv
[17/2352] skipped_exists: 2013-07-22_mN71ltMREcw.mkv
[18/2352] skipped_exists: 2013-08-25__wQ51d8TFeI.mp4
[19/2352] skipped_exists: 2013-09-12_bXohZYOu28o.mkv
[2

,index,input_file,output_file,status,returncode,output_exists,stderr_tail
0,1,downloads\2012-11-15_wtsSeRrOBZQ\2012-11-15_wt...,normalized_mp4\2012-11-15_wtsSeRrOBZQ.mp4,skipped_exists,NaN,True,NaN
1,2,downloads\2013-04-26_8HHUta7n0y4\2013-04-26_8H...,normalized_mp4\2013-04-26_8HHUta7n0y4.mp4,skipped_exists,NaN,True,NaN
2,3,downloads\2013-04-26_bzkLtFLpWEg\2013-04-26_bz...,normalized_mp4\2013-04-26_bzkLtFLpWEg.mp4,skipped_exists,NaN,True,NaN
3,4,downloads\2013-04-26_eDrIDaJRwJ0\2013-04-26_eD...,normalized_mp4\2013-04-26_eDrIDaJRwJ0.mp4,skipped_exists,NaN,True,NaN
4,5,downloads\2013-04-26_OKz_pgqQycc\2013-04-26_OK...,normalized_mp4\2013-04-26_OKz_pgqQycc.mp4,skipped_exists,NaN,True,NaN


Verify output

In [5]:
normalized_files = find_video_files(OUTPUT_ROOT, allowed_exts={".mp4"})
print(f"Normalized mp4 files found: {len(normalized_files)}")

normalized_rows = []
for vf in normalized_files:
    probe = probe_streams(vf)
    summary = summarize_probe(probe)
    normalized_rows.append({
        "output_path": str(vf),
        "suffix": vf.suffix.lower(),
        **summary
    })

normalized_df = pd.DataFrame(normalized_rows)
normalized_df.head()

Normalized mp4 files found: 1421


,output_path,suffix,duration,size,bit_rate,video_codec,audio_codec,width,height,audio_channels,sample_rate
0,normalized_mp4\2012-11-15_wtsSeRrOBZQ.mp4,.mp4,104.600000,22150991,1694148,h264,aac,854.0,480.0,2.0,96000
1,normalized_mp4\2013-04-26_8HHUta7n0y4.mp4,.mp4,334.607000,52548412,1256361,h264,aac,854.0,480.0,2.0,96000
2,normalized_mp4\2013-04-26_bzkLtFLpWEg.mp4,.mp4,242.800000,19154273,631112,h264,aac,854.0,480.0,2.0,96000
3,normalized_mp4\2013-04-26_eDrIDaJRwJ0.mp4,.mp4,338.900000,48354056,1141435,h264,aac,854.0,480.0,2.0,96000
4,normalized_mp4\2013-04-26_OKz_pgqQycc.mp4,.mp4,438.300000,49490014,903308,h264,aac,854.0,480.0,2.0,96000


In [6]:
print("Output suffix distribution:")
display(normalized_df["suffix"].value_counts(dropna=False))

print("\nOutput video codec distribution:")
display(normalized_df["video_codec"].value_counts(dropna=False))

print("\nOutput audio codec distribution:")
display(normalized_df["audio_codec"].value_counts(dropna=False))

print("\nOutput sample rates:")
display(normalized_df["sample_rate"].value_counts(dropna=False).head(10))

Output suffix distribution:


suffix
.mp4    1421
Name: count, dtype: int64


Output video codec distribution:


video_codec
h264    1420
NaN        1
Name: count, dtype: int64


Output audio codec distribution:


audio_codec
aac    1420
NaN       1
Name: count, dtype: int64


Output sample rates:


sample_rate
96000    1420
NaN         1
Name: count, dtype: int64

In [7]:
normalized_files = find_video_files(OUTPUT_ROOT, allowed_exts={".mp4"})

normalized_rows = []
for vf in normalized_files:
    probe = probe_streams(vf)
    summary = summarize_probe(probe)
    normalized_rows.append({
        "output_path": str(vf),
        **summary
    })

normalized_df = pd.DataFrame(normalized_rows)
normalized_df["width"] = pd.to_numeric(normalized_df["width"], errors="coerce")
normalized_df["height"] = pd.to_numeric(normalized_df["height"], errors="coerce")
normalized_df["long_edge"] = normalized_df[["width", "height"]].max(axis=1)

print("Converted resolution summary:")
display(normalized_df["long_edge"].describe())

print("\nAny outputs above target?")
display((normalized_df["long_edge"] > MAX_LONG_EDGE).value_counts())

print("\nMost common output resolutions:")
display(
    normalized_df.groupby(["width", "height"])
    .size()
    .sort_values(ascending=False)
    .head(20)
)

Converted resolution summary:


count    1420.000000
mean      852.191549
std        19.596132
min       640.000000
25%       854.000000
50%       854.000000
75%       854.000000
max       854.000000
Name: long_edge, dtype: float64


Any outputs above target?


long_edge
False    1421
Name: count, dtype: int64


Most common output resolutions:


width  height
854.0  480.0     1402
640.0  480.0        8
       360.0        4
854.0  364.0        3
       342.0        1
       454.0        1
       640.0        1
dtype: int64

In [8]:
normalized_df["fps"] = normalized_df["bit_rate"]  # placeholder column

def get_fps(path):
    cmd = [
        "ffprobe",
        "-v", "0",
        "-select_streams", "v:0",
        "-show_entries", "stream=r_frame_rate",
        "-of", "default=noprint_wrappers=1:nokey=1",
        str(path)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    return result.stdout.strip()

fps_values = []

for vf in normalized_files:
    fps_values.append(get_fps(vf))

print("Frame rate distribution:")
pd.Series(fps_values).value_counts()

Frame rate distribution:


25/1    1420
           1
Name: count, dtype: int64